# Stage 8 — Cross-Modality Edge-Library Expansion and Protocol Seal

This notebook extends Cross-Modal Diagnostic Observability beyond retinal fundus imaging into two additional task-aligned modality families:

- **Dermoscopic skin lesions:** melanoma versus melanocytic nevus across UDA-1, MSK-1, and HAM10000.
- **Frontal chest radiography:** TB-consistent manifestation versus non-TB across Montgomery, Shenzhen, and TBX11K.

It is a **retrospective cross-modality discovery expansion**, not the final prospective validation of DDO2. The notebook freezes its analysis specification before acquiring new data, uses one fixed ImageNet ResNet50 V2-weight representation and one fixed linear probe, writes source axes before held-out validation, freezes label-free target predictions before outcome evaluation, and does not fit a final DDO2 predictor.

Original images are never copied to Google Drive. Public images are streamed or held only in `/content/stage8_tmp`, converted to frozen embeddings, and deleted at the end. New Drive artefacts are hard-capped at 1 GiB.


In [1]:
#@title 08-0. Mount Drive, verify Stage 7, freeze the Stage 8 protocol, and declare the safety boundary
from google.colab import drive
drive.mount("/content/drive")

import gc
import hashlib
import importlib.metadata
import io
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import time
import warnings
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

import numpy as np
import pandas as pd


print("================ STAGE 8 CROSS-MODAL PREFLIGHT ================")

PROJECT_ROOT = Path("/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability")
CODE_ROOT = PROJECT_ROOT / "05_Code" / "Cross_Modal"
RETINAL_TEST_ROOT = (
    PROJECT_ROOT / "06_Data_Records" / "Retinal_DR" /
    "Prospective_Retinal_Blind_Test_v0.1"
)
STAGE7_ROOT = (
    RETINAL_TEST_ROOT /
    "Stage7_PostUnseal_FourDomain_Transportability_Discovery_And_DDO2_Prototype_v0.1"
)
STAGE7_FINAL_PATH = STAGE7_ROOT / "03_Results" / "Stage7_FourDomain_Discovery_Complete_v0.1.json"
STAGE7_EDGE_PATH = STAGE7_ROOT / "01_Four_Domain_Edge_Matrix" / "Stage7_FourDomain_Edge_Matrix_v0.1.csv"

STAGE8_ROOT = (
    PROJECT_ROOT / "06_Data_Records" / "Cross_Modal" /
    "Stage8_CrossModality_EdgeLibrary_Expansion_v0.1"
)
PROTOCOL_ROOT = STAGE8_ROOT / "00_Protocol"
ACQUISITION_ROOT = STAGE8_ROOT / "01_Acquisition_Manifests"
EMBEDDING_ROOT = STAGE8_ROOT / "02_Frozen_Embeddings"
AXIS_ROOT = STAGE8_ROOT / "03_Frozen_Source_Axes"
FREEZE_ROOT = STAGE8_ROOT / "04_Prediction_Freeze"
DISCOVERY_ROOT = STAGE8_ROOT / "05_Unsealed_Discovery"
RESULT_ROOT = STAGE8_ROOT / "06_Results"
for directory in [
    CODE_ROOT, PROTOCOL_ROOT, ACQUISITION_ROOT, EMBEDDING_ROOT,
    AXIS_ROOT, FREEZE_ROOT, DISCOVERY_ROOT, RESULT_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = CODE_ROOT / "CrossModal_Stage8_CrossModality_EdgeLibrary_Expansion_And_Protocol_Seal_v0.1.ipynb"
PROTOCOL_SEAL_PATH = PROTOCOL_ROOT / "Stage8_CrossModality_Expansion_Protocol_Seal_v0.1.json"
DATASET_REGISTRY_PATH = PROTOCOL_ROOT / "Stage8_Frozen_Dataset_Registry_v0.1.csv"
INPUT_COMMITMENT_PATH = PROTOCOL_ROOT / "Stage8_Input_Integrity_Commitment_v0.1.csv"
RUNTIME_STATE_PATH = RESULT_ROOT / "Stage8_Runtime_State_v0.1.json"
FINAL_RECORD_PATH = RESULT_ROOT / "Stage8_CrossModality_Expansion_Complete_v0.1.json"
TEMP_ROOT = Path("/content/stage8_tmp")
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_STAGE7_FINAL_HASH = "293db1a6c41f86bcd4c94d91c2d6ad7dfdb5a9369fc4312beacce73b914cd4be"
USER_REPORTED_FREE_DRIVE_GB = 8.06
MAXIMUM_NEW_STAGE8_BYTES = 1024 * 1024 * 1024
RANDOM_SEED = 20260721
N_BOOTSTRAP = 400
FROZEN_FEATURE_DIMENSION = 2048
MAX_RECORDS_PER_DATASET = 2000
FIXED_THRESHOLD = 0.50
AUC_MINIMUM = 0.70
AUC_CI_LOWER_STRICT_MINIMUM = 0.55
CALIBRATION_DEGRADATION_TOLERANCE = 0.05
OPERATING_POINT_BALANCED_ACCURACY_MINIMUM = 0.70
DOWNLOAD_WORKERS = 8
EMBEDDING_BATCH_SIZE = 24


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def sha256_bytes(raw):
    return hashlib.sha256(raw).hexdigest()


def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sha256_json(payload):
    raw = json.dumps(
        payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def atomic_json(path, payload):
    temporary = Path(str(path) + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
    os.replace(temporary, path)


def write_csv(path, frame):
    text = frame.to_csv(index=False, lineterminator="\n", float_format="%.12g")
    if path.is_file():
        assert path.read_text(encoding="utf-8") == text, f"Existing immutable CSV differs: {path}"
    else:
        path.write_text(text, encoding="utf-8")


def write_progress_csv(path, frame):
    """Permit checkpoint progress to advance until predictions are frozen."""
    text = frame.to_csv(index=False, lineterminator="\n", float_format="%.12g")
    prediction_freeze = FREEZE_ROOT / "Stage8_Prediction_Freeze_Complete_v0.1.json"
    if prediction_freeze.is_file() or FINAL_RECORD_PATH.is_file():
        assert path.is_file() and path.read_text(encoding="utf-8") == text, (
            f"Frozen CSV differs on rerun: {path}"
        )
        return
    temporary = Path(str(path) + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def normalised_notebook_source_sha256(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        notebook = json.load(handle)
    payload = []
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") not in {"code", "markdown"}:
            continue
        value = cell.get("source", [])
        value = "".join(value) if isinstance(value, list) else str(value)
        payload.append({
            "cell_type": cell["cell_type"],
            "source": value.replace("\r\n", "\n"),
        })
    return sha256_json(payload)


assert NOTEBOOK_PATH.is_file(), f"Notebook must be opened from its frozen Drive location: {NOTEBOOK_PATH}"
assert STAGE7_FINAL_PATH.is_file() and STAGE7_EDGE_PATH.is_file()
with STAGE7_FINAL_PATH.open("r", encoding="utf-8") as handle:
    stage7_final = json.load(handle)
stage7_claim = stage7_final["final_record_sha256"]
stage7_without_claim = dict(stage7_final)
stage7_without_claim.pop("final_record_sha256")
assert sha256_json(stage7_without_claim) == stage7_claim
assert stage7_claim == EXPECTED_STAGE7_FINAL_HASH
assert stage7_final["decision"] == "GO_EXPAND_DATASETS_THEN_FIT_DDO2_USING_DISCOVERED_THREE_AXIS_COMPONENTS"

DATASET_REGISTRY = pd.DataFrame([
    {
        "dataset": "ISIC_UDA1", "modality": "dermoscopy", "task": "melanoma_vs_melanocytic_nevus",
        "provider": "ISIC Archive", "official_id": "collection_292", "expected_images": 557,
        "access_url": "https://api.isic-archive.com/collections/292/",
        "image_access": "public_ISIC_S3", "persistent_images_on_drive": False,
    },
    {
        "dataset": "ISIC_MSK1", "modality": "dermoscopy", "task": "melanoma_vs_melanocytic_nevus",
        "provider": "ISIC Archive", "official_id": "collection_289", "expected_images": 1678,
        "access_url": "https://api.isic-archive.com/collections/289/",
        "image_access": "public_ISIC_S3", "persistent_images_on_drive": False,
    },
    {
        "dataset": "HAM10000", "modality": "dermoscopy", "task": "melanoma_vs_melanocytic_nevus",
        "provider": "ISIC Archive", "official_id": "collection_66", "expected_images": 10015,
        "access_url": "https://api.isic-archive.com/collections/66/",
        "image_access": "public_ISIC_S3", "persistent_images_on_drive": False,
    },
    {
        "dataset": "Montgomery_CXR", "modality": "chest_radiography", "task": "tb_manifestation_vs_non_tb",
        "provider": "US National Library of Medicine", "official_id": "Montgomery_County_CXR_Set", "expected_images": 138,
        "access_url": "https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/MontgomerySet/CXR_png/",
        "image_access": "public_NLM_directory", "persistent_images_on_drive": False,
    },
    {
        "dataset": "Shenzhen_CXR", "modality": "chest_radiography", "task": "tb_manifestation_vs_non_tb",
        "provider": "US National Library of Medicine", "official_id": "Shenzhen_Hospital_CXR_Set", "expected_images": 662,
        "access_url": "https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Shenzhen-Hospital-CXR-Set/CXR_png/",
        "image_access": "public_NLM_directory", "persistent_images_on_drive": False,
    },
    {
        "dataset": "TBX11K", "modality": "chest_radiography", "task": "tb_manifestation_vs_non_tb",
        "provider": "TBX11K authors", "official_id": "google_drive_1r-oNYTPiPCOUzSjChjCIYTdkjBTugqxR", "expected_images": 11200,
        "access_url": "https://drive.google.com/file/d/1r-oNYTPiPCOUzSjChjCIYTdkjBTugqxR/view",
        "image_access": "author_public_Google_Drive", "persistent_images_on_drive": False,
    },
])

ANALYSIS_SPEC = {
    "scope": "retrospective cross-modality discovery expansion; not final prospective DDO2 validation",
    "modality_families": {
        "dermoscopy": {
            "endpoint": "melanoma versus melanocytic nevus only",
            "datasets": ["ISIC_UDA1", "ISIC_MSK1", "HAM10000"],
        },
        "chest_radiography": {
            "endpoint": "TB-consistent manifestation versus non-TB",
            "datasets": ["Montgomery_CXR", "Shenzhen_CXR", "TBX11K"],
        },
    },
    "representation": "fixed torchvision ResNet50 IMAGENET1K_V2 global-average-pooled 2048D L2 embeddings",
    "source_probe": "StandardScaler plus class-balanced L2 logistic regression C=1 liblinear",
    "source_gate": "development grouped OOF and held-out validation AUC each >=0.70 with bootstrap lower 95% CI >0.55",
    "target_evaluation": "fixed target validation cohort; no target refit, threshold tuning, sign reversal, or feature selection",
    "failure_axes": ["discrimination", "calibration", "operating_point"],
    "precommitted_stage7_relations": {
        "target_auc__target_mean_knn_distance": "negative",
        "source_minus_target_auc__atc_estimated_accuracy": "positive",
        "target_ece10__unlabeled_mixture_prevalence": "negative",
        "calibration_ece_degradation__atc_estimated_accuracy": "positive",
        "target_balanced_accuracy_at_0_5__unlabeled_mixture_prevalence": "positive",
    },
    "method_boundary": "no final DDO2 fit; only edge-library construction and sign replication",
    "sampling": f"deterministic class-blind hash cap of {MAX_RECORDS_PER_DATASET} eligible records per dataset",
    "storage": "images temporary only; Drive receives manifests, embeddings, axes, predictions, tables, records, and figures; 1 GiB hard cap",
    "access_failure_policy": "record unavailable dataset and continue; never substitute an unapproved mirror or incompatible endpoint",
}

environment = {
    "python": platform.python_version(),
    "numpy": importlib.metadata.version("numpy"),
    "pandas": importlib.metadata.version("pandas"),
}
seal_payload = {
    "stage": "Stage8",
    "decision": "SEALED_CROSS_MODALITY_EDGE_LIBRARY_EXPANSION_PROTOCOL",
    "stage7_final_record_sha256": stage7_claim,
    "notebook_source_sha256": normalised_notebook_source_sha256(NOTEBOOK_PATH),
    "analysis_spec": ANALYSIS_SPEC,
    "analysis_spec_sha256": sha256_json(ANALYSIS_SPEC),
    "dataset_registry_sha256": sha256_json(json.loads(DATASET_REGISTRY.to_json(orient="records"))),
    "environment": environment,
    "user_reported_free_drive_gb": USER_REPORTED_FREE_DRIVE_GB,
    "maximum_new_stage8_bytes": MAXIMUM_NEW_STAGE8_BYTES,
    "sealed_utc": utc_now(),
}
seal_payload["seal_sha256"] = sha256_json(seal_payload)
if PROTOCOL_SEAL_PATH.is_file():
    with PROTOCOL_SEAL_PATH.open("r", encoding="utf-8") as handle:
        existing = json.load(handle)
    existing_without_claim = dict(existing)
    existing_claim = existing_without_claim.pop("seal_sha256")
    assert sha256_json(existing_without_claim) == existing_claim
    assert existing["stage7_final_record_sha256"] == stage7_claim
    assert existing["notebook_source_sha256"] == seal_payload["notebook_source_sha256"]
    assert existing["analysis_spec_sha256"] == seal_payload["analysis_spec_sha256"]
    seal_payload = existing
else:
    atomic_json(PROTOCOL_SEAL_PATH, seal_payload)

write_csv(DATASET_REGISTRY_PATH, DATASET_REGISTRY)
input_commitment = pd.DataFrame([
    {"relative_role": "stage7_final_record", "path": str(STAGE7_FINAL_PATH), "size_bytes": STAGE7_FINAL_PATH.stat().st_size, "sha256": sha256_file(STAGE7_FINAL_PATH)},
    {"relative_role": "stage7_edge_matrix", "path": str(STAGE7_EDGE_PATH), "size_bytes": STAGE7_EDGE_PATH.stat().st_size, "sha256": sha256_file(STAGE7_EDGE_PATH)},
    {"relative_role": "stage8_notebook", "path": str(NOTEBOOK_PATH), "size_bytes": NOTEBOOK_PATH.stat().st_size, "sha256": normalised_notebook_source_sha256(NOTEBOOK_PATH)},
]).sort_values("relative_role").reset_index(drop=True)
write_csv(INPUT_COMMITMENT_PATH, input_commitment)

runtime_state = {
    "stage8_protocol_seal_sha256": seal_payload["seal_sha256"],
    "stage7_final_record_sha256": stage7_claim,
    "scope": "RETROSPECTIVE_CROSS_MODALITY_DISCOVERY_EXPANSION",
    "external_data_downloaded": False,
    "image_files_read": False,
    "image_files_copied_to_drive": False,
    "target_model_refit": False,
    "threshold_tuned": False,
    "final_ddo2_fitted": False,
    "last_updated_utc": utc_now(),
}
atomic_json(RUNTIME_STATE_PATH, runtime_state)

print("Stage 7 final record verified:", stage7_claim)
print("Stage 8 protocol seal:", seal_payload["seal_sha256"])
print("Frozen datasets:", len(DATASET_REGISTRY))
print("User-reported free Drive (GB):", USER_REPORTED_FREE_DRIVE_GB)
print("New Stage 8 Drive cap (GB):", MAXIMUM_NEW_STAGE8_BYTES / 1024**3)
print("Final DDO2 fitting authorised: False")


Mounted at /content/drive
================ STAGE 8 CROSS-MODAL PREFLIGHT ================
Stage 7 final record verified: 293db1a6c41f86bcd4c94d91c2d6ad7dfdb5a9369fc4312beacce73b914cd4be
Stage 8 protocol seal: 731c5cf6b950d2d58873c130de8cbd1eb57d6932379f549a4adfb976aa46c34b
Frozen datasets: 6
User-reported free Drive (GB): 8.06
New Stage 8 Drive cap (GB): 1.0
Final DDO2 fitting authorised: False


In [2]:
#@title 08-1. Acquire harmonised manifests from official public sources without storing images on Drive
import requests

try:
    import gdown
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown


SESSION_HEADERS = {"User-Agent": "Mozilla/5.0 CDO-Stage8-Research/0.1"}


def request_bytes(url, attempts=4, timeout=(20, 180)):
    error = None
    for attempt in range(attempts):
        try:
            response = requests.get(url, headers=SESSION_HEADERS, timeout=timeout)
            response.raise_for_status()
            return response.content
        except Exception as exc:
            error = exc
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Failed after {attempts} attempts: {url}: {error}")


def deterministic_cap(frame, dataset, limit):
    if len(frame) <= limit:
        return frame.sort_values("image_id").reset_index(drop=True)
    ranking = frame["image_id"].map(
        lambda value: hashlib.sha256(f"{RANDOM_SEED}|{dataset}|{value}".encode()).hexdigest()
    )
    return frame.assign(_rank=ranking).sort_values("_rank").head(limit).drop(columns="_rank").reset_index(drop=True)


def find_first_column(frame, candidates):
    lookup = {str(column).strip().lower(): column for column in frame.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None


def harmonise_isic(dataset, collection_id):
    metadata_url = f"https://api.isic-archive.com/collections/{collection_id}/metadata/"
    raw = request_bytes(metadata_url)
    metadata = pd.read_csv(io.BytesIO(raw), low_memory=False)
    image_column = find_first_column(metadata, ["isic_id", "image_name", "image", "name"])
    if image_column is None:
        raise ValueError(f"No ISIC image identifier in columns: {metadata.columns.tolist()}")
    diagnosis_columns = [
        column for column in metadata.columns
        if "diagnos" in str(column).lower() or "benign_malignant" in str(column).lower()
    ]
    if not diagnosis_columns:
        raise ValueError("No diagnosis columns in ISIC metadata")
    diagnosis_text = metadata[diagnosis_columns].fillna("").astype(str).agg(" | ".join, axis=1).str.lower()
    positive = diagnosis_text.str.contains("melanoma", regex=False)
    negative = diagnosis_text.str.contains(r"\b(?:nevus|naevus)\b", regex=True) & ~positive
    eligible = positive | negative
    frame = metadata.loc[eligible].copy()
    frame["label"] = positive.loc[eligible].astype(int).to_numpy()
    frame["image_id"] = frame[image_column].astype(str).str.strip()

    lesion_column = find_first_column(frame, ["lesion_id", "lesion", "lesion_uid"])
    patient_column = find_first_column(frame, ["patient_id", "patient", "patient_uid"])
    lesion = frame[lesion_column].astype(str) if lesion_column else pd.Series("", index=frame.index)
    patient = frame[patient_column].astype(str) if patient_column else pd.Series("", index=frame.index)
    lesion = lesion.where(~lesion.str.lower().isin(["", "nan", "none"]), frame["image_id"])
    patient = patient.where(~patient.str.lower().isin(["", "nan", "none"]), lesion)

    result = pd.DataFrame({
        "dataset": dataset,
        "modality": "dermoscopy",
        "task": "melanoma_vs_melanocytic_nevus",
        "image_id": frame["image_id"],
        "unit_id": dataset + "::LESION::" + lesion.astype(str),
        "group_id": dataset + "::PATIENT::" + patient.astype(str),
        "label": frame["label"].astype(int),
        "source_locator": frame["image_id"].map(lambda value: f"https://isic-archive.s3.amazonaws.com/images/{value}.jpg"),
        "source_kind": "https",
        "official_record_scope": f"ISIC_collection_{collection_id}",
    }).drop_duplicates("image_id")
    result = deterministic_cap(result, dataset, MAX_RECORDS_PER_DATASET)
    counts = result["label"].value_counts()
    if not {0, 1}.issubset(counts.index) or counts.min() < 20:
        raise ValueError(f"Insufficient harmonised class counts for {dataset}: {counts.to_dict()}")
    return result


def harmonise_nlm(dataset, directory_url):
    html = request_bytes(directory_url).decode("utf-8", errors="replace")
    names = sorted(set(re.findall(r'href=["\']([^"\']+\.png)["\']', html, flags=re.I)))
    if not names:
        names = sorted(set(re.findall(r'([A-Za-z0-9_-]+_[01]\.png)', html, flags=re.I)))
    rows = []
    for name in names:
        match = re.search(r"_([01])\.png$", name, flags=re.I)
        if match is None:
            continue
        rows.append({
            "dataset": dataset,
            "modality": "chest_radiography",
            "task": "tb_manifestation_vs_non_tb",
            "image_id": Path(name).stem,
            "unit_id": dataset + "::IMAGE::" + Path(name).stem,
            "group_id": dataset + "::PATIENT::" + Path(name).stem,
            "label": int(match.group(1)),
            "source_locator": urljoin(directory_url, name),
            "source_kind": "https",
            "official_record_scope": "NLM_public_TB_CXR",
        })
    result = pd.DataFrame(rows).drop_duplicates("image_id")
    counts = result["label"].value_counts() if len(result) else pd.Series(dtype=int)
    if len(result) < 100 or not {0, 1}.issubset(counts.index):
        raise ValueError(f"Unexpected NLM manifest for {dataset}: n={len(result)}, classes={counts.to_dict()}")
    return result.sort_values("image_id").reset_index(drop=True)


def infer_tbx_label(path):
    text = str(path).replace("\\", "/").lower()
    parts = [part for part in text.split("/") if part]
    if any("uncertain" in part for part in parts):
        return None
    if any(part in {"healthy", "health", "normal"} for part in parts):
        return 0
    if any(("sick" in part and "non" in part) or part in {"sick", "non-tb", "nontb"} for part in parts):
        return 0
    if any(part in {"tb", "active_tb", "latent_tb", "active&latent_tb", "active_latent_tb"} for part in parts):
        return 1
    stem = Path(path).stem.lower()
    if re.match(r"^(active|latent|tb)[_-]", stem):
        return 1
    if re.match(r"^(healthy|health|normal|sick)[_-]", stem):
        return 0
    return None


def harmonise_tbx11k():
    archive_path = TEMP_ROOT / "TBX11K.zip"
    extract_root = TEMP_ROOT / "TBX11K_extracted"
    if not archive_path.is_file():
        result = gdown.download(
            id="1r-oNYTPiPCOUzSjChjCIYTdkjBTugqxR",
            output=str(archive_path), quiet=False, fuzzy=True,
        )
        if result is None or not archive_path.is_file():
            raise RuntimeError("Official TBX11K Google Drive download did not complete")
    if not extract_root.exists():
        extract_root.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(archive_path), str(extract_root))

    allowed_basenames = set()
    for text_path in extract_root.rglob("*.txt"):
        if not any(token in text_path.name.lower() for token in ["train", "val"]):
            continue
        try:
            for line in text_path.read_text(encoding="utf-8", errors="ignore").splitlines():
                for token in re.findall(r"[^\s,]+\.(?:png|jpg|jpeg)", line, flags=re.I):
                    allowed_basenames.add(Path(token).name.lower())
        except Exception:
            pass

    image_paths = sorted([
        path for path in extract_root.rglob("*")
        if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg"}
    ])
    rows = []
    for path in image_paths:
        if allowed_basenames and path.name.lower() not in allowed_basenames:
            continue
        label = infer_tbx_label(path.relative_to(extract_root))
        if label is None:
            continue
        rows.append({
            "dataset": "TBX11K",
            "modality": "chest_radiography",
            "task": "tb_manifestation_vs_non_tb",
            "image_id": path.stem,
            "unit_id": "TBX11K::IMAGE::" + path.stem,
            "group_id": "TBX11K::PATIENT::" + path.stem,
            "label": int(label),
            "source_locator": str(path),
            "source_kind": "local_temporary",
            "official_record_scope": "TBX11K_author_train_and_validation",
        })
    result = pd.DataFrame(rows).drop_duplicates("image_id")
    result = deterministic_cap(result, "TBX11K", MAX_RECORDS_PER_DATASET)
    counts = result["label"].value_counts() if len(result) else pd.Series(dtype=int)
    if len(result) < 500 or not {0, 1}.issubset(counts.index) or counts.min() < 50:
        raise ValueError(f"Could not recover reliable TBX11K train/validation labels: n={len(result)}, classes={counts.to_dict()}")
    return result


builders = {
    "ISIC_UDA1": lambda: harmonise_isic("ISIC_UDA1", 292),
    "ISIC_MSK1": lambda: harmonise_isic("ISIC_MSK1", 289),
    "HAM10000": lambda: harmonise_isic("HAM10000", 66),
    "Montgomery_CXR": lambda: harmonise_nlm(
        "Montgomery_CXR",
        "https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/MontgomerySet/CXR_png/",
    ),
    "Shenzhen_CXR": lambda: harmonise_nlm(
        "Shenzhen_CXR",
        "https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Shenzhen-Hospital-CXR-Set/CXR_png/",
    ),
    "TBX11K": harmonise_tbx11k,
}

dataset_manifests = {}
acquisition_rows = []
for dataset in DATASET_REGISTRY["dataset"]:
    manifest_path = ACQUISITION_ROOT / f"{dataset}_Harmonised_Acquisition_Manifest_v0.1.csv"
    try:
        # Only a genuinely complete checkpoint may reuse a manifest whose
        # temporary local image paths no longer exist after cleanup.
        checkpoint_path = EMBEDDING_ROOT / f"{dataset}_Frozen_ResNet50V2_L2_Embeddings_v0.1.npz"
        checkpoint_complete = False
        if checkpoint_path.is_file():
            with np.load(checkpoint_path, allow_pickle=False) as saved_checkpoint:
                checkpoint_complete = bool(
                    len(saved_checkpoint["completed"]) > 0 and
                    np.all(saved_checkpoint["completed"].astype(np.int8) == 1)
                )
        if manifest_path.is_file() and checkpoint_complete:
            manifest = pd.read_csv(manifest_path)
        else:
            manifest = builders[dataset]()
            write_csv(manifest_path, manifest)
        dataset_manifests[dataset] = manifest
        counts = manifest["label"].value_counts().to_dict()
        acquisition_rows.append({
            "dataset": dataset, "status": "READY", "records": len(manifest),
            "negative": int(counts.get(0, 0)), "positive": int(counts.get(1, 0)),
            "manifest_sha256": sha256_file(manifest_path), "error": "",
        })
        print(f"{dataset}: READY n={len(manifest)} classes={counts}")
    except Exception as exc:
        acquisition_rows.append({
            "dataset": dataset, "status": "UNAVAILABLE", "records": 0,
            "negative": 0, "positive": 0, "manifest_sha256": "", "error": repr(exc)[:1000],
        })
        print(f"{dataset}: UNAVAILABLE -> {exc}")

acquisition_status = pd.DataFrame(acquisition_rows)
write_progress_csv(ACQUISITION_ROOT / "Stage8_Dataset_Acquisition_Status_v0.1.csv", acquisition_status)
display(acquisition_status)

runtime_state.update({
    "external_data_downloaded": True,
    "datasets_ready_after_acquisition": acquisition_status.loc[acquisition_status["status"].eq("READY"), "dataset"].tolist(),
    "last_updated_utc": utc_now(),
})
atomic_json(RUNTIME_STATE_PATH, runtime_state)


ISIC_UDA1: READY n=554 classes={0: 395, 1: 159}
ISIC_MSK1: READY n=1193 classes={0: 820, 1: 373}
HAM10000: READY n=2000 classes={0: 1708, 1: 292}
Montgomery_CXR: UNAVAILABLE -> Failed after 4 attempts: https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/MontgomerySet/CXR_png/: 403 Client Error: Forbidden for url: https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/MontgomerySet/CXR_png/
Shenzhen_CXR: UNAVAILABLE -> Failed after 4 attempts: https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Shenzhen-Hospital-CXR-Set/CXR_png/: 403 Client Error: Forbidden for url: https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Shenzhen-Hospital-CXR-Set/CXR_png/


Downloading...
From (original): https://drive.google.com/uc?id=1r-oNYTPiPCOUzSjChjCIYTdkjBTugqxR
From (redirected): https://drive.google.com/uc?id=1r-oNYTPiPCOUzSjChjCIYTdkjBTugqxR&confirm=t&uuid=b5e69a5c-e4b8-4639-b60f-78cc64483eb2
To: /content/stage8_tmp/TBX11K.zip
100%|██████████| 3.31G/3.31G [00:58<00:00, 56.4MB/s]


TBX11K: READY n=2000 classes={0: 1827, 1: 173}


,dataset,status,records,negative,positive,manifest_sha256,error
0,ISIC_UDA1,READY,554,395,159,4c0fb4aa6966cc55b0a8d359e47daa8a20e76172ec6f88...,
1,ISIC_MSK1,READY,1193,820,373,df98282397d005e7caa5444c1d734bea5b85724051f34b...,
2,HAM10000,READY,2000,1708,292,2743942fc27e92dc24770f9f58e8c18ff159901adcbaac...,
3,Montgomery_CXR,UNAVAILABLE,0,0,0,,RuntimeError('Failed after 4 attempts: https:/...
4,Shenzhen_CXR,UNAVAILABLE,0,0,0,,RuntimeError('Failed after 4 attempts: https:/...
5,TBX11K,READY,2000,1827,173,6264fe1a20b2233520b49521d48b51cc048868c20cb4d6...,


In [3]:
#@title 08-2. Stream images, extract fixed ResNet50 V2-weight embeddings, checkpoint, and audit duplicates
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
from torchvision.models import ResNet50_Weights, resnet50
from tqdm.auto import tqdm


torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = ResNet50_Weights.IMAGENET1K_V2
transform = weights.transforms()
backbone = resnet50(weights=weights)
backbone.fc = torch.nn.Identity()
backbone.eval().to(DEVICE)
for parameter in backbone.parameters():
    parameter.requires_grad_(False)

print("Execution device:", DEVICE)
print("Backbone: torchvision ResNet50 IMAGENET1K_V2, 2048D L2")


def difference_hash(image, size=16):
    gray = image.convert("L").resize((size + 1, size), Image.Resampling.BILINEAR)
    values = np.asarray(gray, dtype=np.uint8)
    bits = values[:, 1:] > values[:, :-1]
    packed = np.packbits(bits.reshape(-1))
    return packed.tobytes().hex()


def read_image_record(record):
    locator = record["source_locator"]
    if record["source_kind"] == "https":
        raw = request_bytes(locator, attempts=4, timeout=(20, 120))
    else:
        raw = Path(locator).read_bytes()
    image = Image.open(io.BytesIO(raw)).convert("RGB")
    return transform(image), sha256_bytes(raw), difference_hash(image)


def atomic_npz(path, **arrays):
    temporary = Path(str(path) + ".tmp.npz")
    np.savez_compressed(temporary, **arrays)
    os.replace(temporary, path)


def extract_dataset_embeddings(dataset, manifest):
    checkpoint_path = EMBEDDING_ROOT / f"{dataset}_Frozen_ResNet50V2_L2_Embeddings_v0.1.npz"
    manifest_hash = sha256_file(ACQUISITION_ROOT / f"{dataset}_Harmonised_Acquisition_Manifest_v0.1.csv")
    ids = manifest["image_id"].astype(str).to_numpy(dtype=str)
    n = len(manifest)
    if checkpoint_path.is_file():
        saved = np.load(checkpoint_path, allow_pickle=False)
        assert str(saved["manifest_sha256"].item()) == manifest_hash
        assert np.array_equal(saved["image_id"].astype(str), ids)
        embeddings = saved["embedding"].astype(np.float32)
        completed = saved["completed"].astype(np.int8)
        content_sha = saved["content_sha256"].astype(str)
        dhash = saved["difference_hash"].astype(str)
    else:
        embeddings = np.zeros((n, FROZEN_FEATURE_DIMENSION), dtype=np.float32)
        completed = np.zeros(n, dtype=np.int8)
        content_sha = np.full(n, "", dtype="<U64")
        dhash = np.full(n, "", dtype="<U64")

    pending = np.flatnonzero(completed != 1)
    print(f"{dataset}: completed {int(np.sum(completed == 1))}/{n}; pending or retry {len(pending)}")
    for start in tqdm(range(0, len(pending), EMBEDDING_BATCH_SIZE), desc=f"Embedding {dataset}"):
        batch_indices = pending[start:start + EMBEDDING_BATCH_SIZE]
        tensors, valid_indices, shas, hashes = [], [], [], []
        with ThreadPoolExecutor(max_workers=min(DOWNLOAD_WORKERS, len(batch_indices))) as pool:
            futures = {
                pool.submit(read_image_record, manifest.iloc[int(index)]): int(index)
                for index in batch_indices
            }
            results = {}
            for future in as_completed(futures):
                index = futures[future]
                try:
                    results[index] = future.result()
                except Exception as exc:
                    completed[index] = -1
                    print(f"Warning: {dataset}/{ids[index]} decode failed: {exc}")
        for index in batch_indices:
            index = int(index)
            if index in results:
                tensor, sha, image_hash = results[index]
                tensors.append(tensor)
                valid_indices.append(index)
                shas.append(sha)
                hashes.append(image_hash)
        if tensors:
            with torch.inference_mode():
                features = backbone(torch.stack(tensors).to(DEVICE)).float()
                features = torch.nn.functional.normalize(features, p=2, dim=1)
            features = features.cpu().numpy().astype(np.float32)
            embeddings[np.asarray(valid_indices)] = features
            completed[np.asarray(valid_indices)] = 1
            content_sha[np.asarray(valid_indices)] = np.asarray(shas, dtype="<U64")
            dhash[np.asarray(valid_indices)] = np.asarray(hashes, dtype="<U64")
        if start % (EMBEDDING_BATCH_SIZE * 5) == 0 or start + EMBEDDING_BATCH_SIZE >= len(pending):
            atomic_npz(
                checkpoint_path,
                image_id=ids,
                embedding=embeddings,
                completed=completed,
                content_sha256=content_sha,
                difference_hash=dhash,
                manifest_sha256=np.asarray(manifest_hash),
            )
    return checkpoint_path


embedding_paths = {}
for dataset, manifest in dataset_manifests.items():
    try:
        embedding_paths[dataset] = extract_dataset_embeddings(dataset, manifest)
    except Exception as exc:
        print(f"Embedding failed for {dataset}: {exc}")

audit_rows = []
for dataset, path in embedding_paths.items():
    saved = np.load(path, allow_pickle=False)
    manifest = dataset_manifests[dataset].copy().reset_index(drop=True)
    assert np.array_equal(manifest["image_id"].astype(str).to_numpy(), saved["image_id"].astype(str))
    for index in np.flatnonzero(saved["completed"].astype(np.int8) == 1):
        row = manifest.iloc[int(index)]
        audit_rows.append({
            "dataset": dataset, "row_index": int(index), "image_id": str(row["image_id"]),
            "modality": row["modality"], "content_sha256": str(saved["content_sha256"][index]),
            "difference_hash": str(saved["difference_hash"][index]),
        })
embedding_audit = pd.DataFrame(audit_rows)
if len(embedding_audit):
    cross_sha_counts = embedding_audit.groupby("content_sha256")["dataset"].nunique()
    cross_dhash_counts = embedding_audit.groupby("difference_hash")["dataset"].nunique()
    duplicate_sha = set(cross_sha_counts[cross_sha_counts > 1].index)
    duplicate_dhash = set(cross_dhash_counts[cross_dhash_counts > 1].index)
    embedding_audit["cross_dataset_exact_duplicate"] = embedding_audit["content_sha256"].isin(duplicate_sha)
    embedding_audit["cross_dataset_visual_hash_duplicate"] = embedding_audit["difference_hash"].isin(duplicate_dhash)
    embedding_audit["excluded_for_cross_dataset_duplicate"] = (
        embedding_audit["cross_dataset_exact_duplicate"] |
        embedding_audit["cross_dataset_visual_hash_duplicate"]
    )
else:
    embedding_audit = pd.DataFrame(columns=[
        "dataset", "row_index", "image_id", "modality", "content_sha256", "difference_hash",
        "cross_dataset_exact_duplicate", "cross_dataset_visual_hash_duplicate",
        "excluded_for_cross_dataset_duplicate",
    ])
write_progress_csv(EMBEDDING_ROOT / "Stage8_CrossDataset_Duplicate_Audit_v0.1.csv", embedding_audit)
display(
    embedding_audit.groupby("dataset", as_index=False).agg(
        embedded_images=("image_id", "size"),
        excluded_duplicates=("excluded_for_cross_dataset_duplicate", "sum"),
    ) if len(embedding_audit) else embedding_audit
)

runtime_state.update({
    "image_files_read": bool(len(embedding_audit)),
    "image_files_copied_to_drive": False,
    "backbone_inference_run": bool(len(embedding_audit)),
    "embedding_datasets": sorted(embedding_paths),
    "last_updated_utc": utc_now(),
})
atomic_json(RUNTIME_STATE_PATH, runtime_state)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 103MB/s]


Execution device: cuda
Backbone: torchvision ResNet50 IMAGENET1K_V2, 2048D L2
ISIC_UDA1: completed 0/554; pending or retry 554


Embedding ISIC_UDA1:   0%|          | 0/24 [00:00<?, ?it/s]

ISIC_MSK1: completed 0/1193; pending or retry 1193


...

HAM10000: completed 0/2000; pending or retry 2000


Embedding HAM10000:   0%|          | 0/84 [00:00<?, ?it/s]

TBX11K: completed 0/2000; pending or retry 2000


Embedding TBX11K:   0%|          | 0/84 [00:00<?, ?it/s]

,dataset,embedded_images,excluded_duplicates
0,HAM10000,2000,0
1,ISIC_MSK1,1193,0
2,ISIC_UDA1,554,0
3,TBX11K,2000,0


In [4]:
#@title 08-3. Create grouped development/validation partitions, freeze source axes, and apply the recoverability gate
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def aggregate_records(manifest, embeddings):
    work = manifest.copy().reset_index(drop=True)
    work["embedding_index"] = np.arange(len(work))
    records, vectors = [], []
    for unit_id, group in work.groupby("unit_id", sort=True):
        labels = group["label"].astype(int).unique()
        if len(labels) != 1:
            continue
        group_ids = group["group_id"].astype(str).unique()
        vector = embeddings[group["embedding_index"].to_numpy(int)].mean(axis=0)
        norm = np.linalg.norm(vector)
        if not np.isfinite(norm) or norm == 0:
            continue
        vectors.append((vector / norm).astype(np.float32))
        records.append({
            "unit_id": str(unit_id), "group_id": sorted(group_ids)[0],
            "label": int(labels[0]), "images": int(len(group)),
        })
    return pd.DataFrame(records), np.asarray(vectors, dtype=np.float32)


def fixed_group_split(table, seed):
    test_size = 0.40 if len(table) < 300 else 0.25
    splitter = GroupShuffleSplit(n_splits=64, test_size=test_size, random_state=seed)
    for development, validation in splitter.split(table, table["label"], table["group_id"]):
        if table.iloc[development]["label"].nunique() == 2 and table.iloc[validation]["label"].nunique() == 2:
            return np.asarray(development), np.asarray(validation)
    raise ValueError("Could not create a two-class group-disjoint development/validation split")


def create_probe():
    return Pipeline([
        ("standardscaler", StandardScaler()),
        ("logisticregression", LogisticRegression(
            C=1.0, penalty="l2", class_weight="balanced", solver="liblinear",
            max_iter=5000, random_state=RANDOM_SEED,
        )),
    ])


def patient_bootstrap_auc(table, probabilities, seed):
    labels = table["label"].to_numpy(int)
    probabilities = np.asarray(probabilities, dtype=float)
    groups = table["group_id"].astype(str).to_numpy()
    unique_groups = np.unique(groups)
    rng = np.random.default_rng(seed)
    values = []
    attempts = 0
    while len(values) < N_BOOTSTRAP and attempts < N_BOOTSTRAP * 20:
        attempts += 1
        sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        sampled_indices = np.concatenate([np.flatnonzero(groups == group) for group in sampled_groups])
        if np.unique(labels[sampled_indices]).size == 2:
            values.append(roc_auc_score(labels[sampled_indices], probabilities[sampled_indices]))
    if len(values) < N_BOOTSTRAP // 2:
        return [np.nan, np.nan]
    return [float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))]


def save_axis(path, probe, development_manifest_hash, source):
    scaler = probe.named_steps["standardscaler"]
    logistic = probe.named_steps["logisticregression"]
    standardised_coefficient = logistic.coef_.reshape(-1).astype(np.float64)
    intercept = float(logistic.intercept_[0])
    scale = scaler.scale_.astype(np.float64)
    mean = scaler.mean_.astype(np.float64)
    raw_coefficient = standardised_coefficient / scale
    raw_intercept = intercept - float(np.dot(mean / scale, standardised_coefficient))
    candidate = {
        "mean": mean, "scale": scale, "standardised_coefficient": standardised_coefficient,
        "standardised_intercept": intercept, "raw_coefficient": raw_coefficient,
        "raw_intercept": raw_intercept,
    }
    if path.is_file():
        with np.load(path, allow_pickle=False) as saved:
            assert str(saved["source"].item()) == source
            assert str(saved["development_manifest_sha256"].item()) == development_manifest_hash
            existing = {
                "mean": saved["scaler_mean"].astype(np.float64),
                "scale": saved["scaler_scale"].astype(np.float64),
                "standardised_coefficient": saved["standardised_coefficient"].astype(np.float64),
                "standardised_intercept": float(saved["standardised_intercept"].item()),
                "raw_coefficient": saved["raw_coefficient"].astype(np.float64),
                "raw_intercept": float(saved["raw_intercept"].item()),
            }
        for key in candidate:
            assert np.allclose(existing[key], candidate[key], rtol=1e-10, atol=1e-12), (
                f"Existing frozen axis differs on rerun: {source}/{key}"
            )
        return existing
    atomic_npz(
        path,
        source=np.asarray(source),
        scaler_mean=mean,
        scaler_scale=scale,
        standardised_coefficient=standardised_coefficient,
        standardised_intercept=np.asarray(intercept),
        raw_coefficient=raw_coefficient,
        raw_intercept=np.asarray(raw_intercept),
        development_manifest_sha256=np.asarray(development_manifest_hash),
    )
    return candidate


def axis_probability(axis, embeddings):
    embeddings = np.asarray(embeddings, dtype=np.float64)
    standardised_logit = ((embeddings - axis["mean"]) / axis["scale"]) @ axis["standardised_coefficient"] + axis["standardised_intercept"]
    raw_logit = embeddings @ axis["raw_coefficient"] + axis["raw_intercept"]
    maximum_error = float(np.max(np.abs(standardised_logit - raw_logit)))
    assert maximum_error < 1e-8, f"Axis equivalence failure: {maximum_error}"
    return expit(raw_logit), raw_logit, maximum_error


duplicate_lookup = set()
if len(embedding_audit):
    duplicate_lookup = set(
        zip(
            embedding_audit.loc[embedding_audit["excluded_for_cross_dataset_duplicate"], "dataset"],
            embedding_audit.loc[embedding_audit["excluded_for_cross_dataset_duplicate"], "row_index"].astype(int),
        )
    )

domain_assets = {}
inventory_rows = []
for dataset, checkpoint_path in embedding_paths.items():
    saved = np.load(checkpoint_path, allow_pickle=False)
    manifest = dataset_manifests[dataset].copy().reset_index(drop=True)
    valid_indices = [
        int(index) for index in np.flatnonzero(saved["completed"].astype(np.int8) == 1)
        if (dataset, int(index)) not in duplicate_lookup
    ]
    if len(valid_indices) < 40:
        continue
    valid_manifest = manifest.iloc[valid_indices].reset_index(drop=True)
    valid_embeddings = saved["embedding"][valid_indices].astype(np.float32)
    table, vectors = aggregate_records(valid_manifest, valid_embeddings)
    if len(table) < 40 or table["label"].nunique() != 2:
        continue
    development, validation = fixed_group_split(table, RANDOM_SEED + len(domain_assets))
    table["partition"] = ""
    table.loc[development, "partition"] = "development"
    table.loc[validation, "partition"] = "validation"
    assert set(table["partition"]) == {"development", "validation"}
    assert set(table.loc[development, "group_id"]).isdisjoint(set(table.loc[validation, "group_id"]))
    domain_assets[dataset] = {
        "table": table, "embedding": vectors,
        "modality": valid_manifest["modality"].iloc[0], "task": valid_manifest["task"].iloc[0],
    }
    for partition in ["development", "validation"]:
        subset = table[table["partition"].eq(partition)]
        inventory_rows.append({
            "dataset": dataset, "modality": domain_assets[dataset]["modality"],
            "task": domain_assets[dataset]["task"], "partition": partition,
            "units": len(subset), "groups": subset["group_id"].nunique(),
            "negative": int((subset["label"] == 0).sum()), "positive": int((subset["label"] == 1).sum()),
        })

partition_inventory = pd.DataFrame(inventory_rows)
write_progress_csv(ACQUISITION_ROOT / "Stage8_Dataset_Partition_Inventory_v0.1.csv", partition_inventory)
display(partition_inventory)

source_rows = []
axes = {}
source_validation_assets = {}
for source_index, (source, asset) in enumerate(domain_assets.items()):
    table = asset["table"]
    embeddings = asset["embedding"].astype(np.float64)
    development_mask = table["partition"].eq("development").to_numpy()
    validation_mask = table["partition"].eq("validation").to_numpy()
    development_table = table.loc[development_mask].reset_index(drop=True)
    validation_table = table.loc[validation_mask].reset_index(drop=True)
    X_development = embeddings[development_mask]
    y_development = development_table["label"].to_numpy(int)
    groups = development_table["group_id"].to_numpy(str)

    positive_groups = np.unique(groups[y_development == 1]).size
    negative_groups = np.unique(groups[y_development == 0]).size
    folds = min(5, positive_groups, negative_groups)
    if folds < 3:
        source_rows.append({
            "source": source, "modality": asset["modality"], "development_units": len(development_table),
            "validation_units": len(validation_table), "development_oof_auc": np.nan,
            "development_oof_auc_ci_lower": np.nan, "development_oof_auc_ci_upper": np.nan,
            "validation_auc": np.nan, "validation_auc_ci_lower": np.nan, "validation_auc_ci_upper": np.nan,
            "recoverable": False, "failure_reason": "INSUFFICIENT_GROUPS_FOR_THREE_FOLDS",
            "axis_path": "", "axis_sha256": "", "maximum_axis_equivalence_error": np.nan,
        })
        continue

    splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=RANDOM_SEED)
    oof_probability = np.full(len(development_table), np.nan)
    for fold, (training, testing) in enumerate(splitter.split(X_development, y_development, groups), 1):
        probe = create_probe()
        probe.fit(X_development[training], y_development[training])
        oof_probability[testing] = probe.predict_proba(X_development[testing])[:, 1]
    assert np.isfinite(oof_probability).all()
    oof_auc = float(roc_auc_score(y_development, oof_probability))
    oof_ci = patient_bootstrap_auc(development_table, oof_probability, RANDOM_SEED + source_index)

    final_probe = create_probe()
    final_probe.fit(X_development, y_development)
    partition_path = ACQUISITION_ROOT / f"{source}_Frozen_Development_Validation_Unit_Manifest_v0.1.csv"
    write_csv(partition_path, table)
    axis_path = AXIS_ROOT / f"{source}_Frozen_Source_Axis_v0.1.npz"
    axis = save_axis(axis_path, final_probe, sha256_file(partition_path), source)
    axis_sha = sha256_file(axis_path)

    validation_probability, validation_logit, equivalence_error = axis_probability(axis, embeddings[validation_mask])
    validation_auc = float(roc_auc_score(validation_table["label"], validation_probability))
    validation_ci = patient_bootstrap_auc(
        validation_table, validation_probability, RANDOM_SEED + 100 + source_index
    )
    recoverable = bool(
        oof_auc >= AUC_MINIMUM and oof_ci[0] > AUC_CI_LOWER_STRICT_MINIMUM and
        validation_auc >= AUC_MINIMUM and validation_ci[0] > AUC_CI_LOWER_STRICT_MINIMUM
    )
    axes[source] = axis
    validation_table = validation_table.copy()
    validation_table["probability"] = validation_probability
    validation_table["logit"] = validation_logit
    source_validation_assets[source] = validation_table
    source_rows.append({
        "source": source, "modality": asset["modality"],
        "development_units": len(development_table), "validation_units": len(validation_table),
        "development_oof_auc": oof_auc, "development_oof_auc_ci_lower": oof_ci[0],
        "development_oof_auc_ci_upper": oof_ci[1], "validation_auc": validation_auc,
        "validation_auc_ci_lower": validation_ci[0], "validation_auc_ci_upper": validation_ci[1],
        "recoverable": recoverable, "failure_reason": "" if recoverable else "SOURCE_GATE_NOT_MET",
        "axis_path": str(axis_path.relative_to(STAGE8_ROOT)), "axis_sha256": axis_sha,
        "maximum_axis_equivalence_error": equivalence_error,
    })
    print(
        f"{source}: OOF AUC {oof_auc:.4f} [{oof_ci[0]:.4f}, {oof_ci[1]:.4f}] | "
        f"validation {validation_auc:.4f} [{validation_ci[0]:.4f}, {validation_ci[1]:.4f}] | recoverable={recoverable}"
    )

source_recoverability_columns = [
    "source", "modality", "development_units", "validation_units",
    "development_oof_auc", "development_oof_auc_ci_lower", "development_oof_auc_ci_upper",
    "validation_auc", "validation_auc_ci_lower", "validation_auc_ci_upper", "recoverable",
    "failure_reason", "axis_path", "axis_sha256", "maximum_axis_equivalence_error",
]
source_recoverability = pd.DataFrame(source_rows, columns=source_recoverability_columns)
write_progress_csv(AXIS_ROOT / "Stage8_Source_Recoverability_Summary_v0.1.csv", source_recoverability)
display(source_recoverability)


,dataset,modality,task,partition,units,groups,negative,positive
0,ISIC_UDA1,dermoscopy,melanoma_vs_melanocytic_nevus,development,415,415,289,126
1,ISIC_UDA1,dermoscopy,melanoma_vs_melanocytic_nevus,validation,139,139,106,33
2,ISIC_MSK1,dermoscopy,melanoma_vs_melanocytic_nevus,development,489,413,353,136
3,ISIC_MSK1,dermoscopy,melanoma_vs_melanocytic_nevus,validation,157,138,110,47
4,HAM10000,dermoscopy,melanoma_vs_melanocytic_nevus,development,1398,1398,1200,198
5,HAM10000,dermoscopy,melanoma_vs_melanocytic_nevus,validation,466,466,402,64
6,TBX11K,chest_radiography,tb_manifestation_vs_non_tb,development,1500,1500,1374,126
7,TBX11K,chest_radiography,tb_manifestation_vs_non_tb,validation,500,500,453,47


ISIC_UDA1: OOF AUC 0.7472 [0.6932, 0.7949] | validation 0.7207 [0.6008, 0.8313] | recoverable=True
ISIC_MSK1: OOF AUC 0.7737 [0.7258, 0.8200] | validation 0.7012 [0.6039, 0.7870] | recoverable=True
HAM10000: OOF AUC 0.8502 [0.8247, 0.8779] | validation 0.8137 [0.7500, 0.8757] | recoverable=True
TBX11K: OOF AUC 0.9588 [0.9408, 0.9751] | validation 0.9638 [0.9397, 0.9833] | recoverable=True


,source,modality,development_units,validation_units,development_oof_auc,development_oof_auc_ci_lower,development_oof_auc_ci_upper,validation_auc,validation_auc_ci_lower,validation_auc_ci_upper,recoverable,failure_reason,axis_path,axis_sha256,maximum_axis_equivalence_error
0,ISIC_UDA1,dermoscopy,415,139,0.747240,0.693216,0.794872,0.720698,0.600825,0.831265,True,,03_Frozen_Source_Axes/ISIC_UDA1_Frozen_Source_...,316857360dd3a4f3c9f19f811cc58a4c17c1da1372319c...,1.598721e-14
1,ISIC_MSK1,dermoscopy,489,157,0.773746,0.725815,0.820006,0.701161,0.603893,0.787039,True,,03_Frozen_Source_Axes/ISIC_MSK1_Frozen_Source_...,5011be88de7e877d836f0518540c53dd3f6a34c57225b7...,1.598721e-14
2,HAM10000,dermoscopy,1398,466,0.850231,0.824684,0.877929,0.813744,0.749970,0.875653,True,,03_Frozen_Source_Axes/HAM10000_Frozen_Source_A...,0988518fcfbcb3436f43fbdff37b61395b8ba8b3715fcc...,2.842171e-14
3,TBX11K,chest_radiography,1500,500,0.958827,0.940773,0.975102,0.963788,0.939730,0.983316,True,,03_Frozen_Source_Axes/TBX11K_Frozen_Source_Axi...,6fc63f746731f607173d7aff4ffdfb2e580be2a4abe5b6...,1.643130e-14


In [5]:
#@title 08-4. Score same-modality targets label-free and freeze predictions before outcome evaluation
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from scipy.stats import wasserstein_distance
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import NearestNeighbors


def deterministic_indices(ids, limit, salt):
    ranked = sorted(
        range(len(ids)),
        key=lambda index: hashlib.sha256(f"{salt}|{ids[index]}".encode()).hexdigest(),
    )
    return np.asarray(ranked[:min(limit, len(ranked))], dtype=int)


def support_components(source_embeddings, target_embeddings):
    neighbors = min(6, len(source_embeddings))
    model = NearestNeighbors(n_neighbors=neighbors, metric="cosine").fit(source_embeddings)
    source_distances = model.kneighbors(source_embeddings, return_distance=True)[0]
    source_reference = source_distances[:, 1:].mean(axis=1)
    target_distances = model.kneighbors(
        target_embeddings, n_neighbors=min(5, len(source_embeddings)), return_distance=True
    )[0].mean(axis=1)
    q95, q99 = np.quantile(source_reference, [0.95, 0.99])
    median_reference = float(np.median(source_reference))
    return {
        "support_fraction": float(np.mean(target_distances <= q95)),
        "fraction_beyond_source_q99": float(np.mean(target_distances > q99)),
        "source_knn_q95": float(q95), "source_knn_q99": float(q99),
        "target_mean_knn_distance": float(target_distances.mean()),
        "target_mean_knn_distance_normalised": float(target_distances.mean() / max(median_reference, 1e-8)),
    }


def geometry_components(source_table, source_embeddings, target_table, target_embeddings, salt):
    limit = min(128, len(source_table), len(target_table))
    source_index = deterministic_indices(source_table["unit_id"].tolist(), limit, salt + "|S")
    target_index = deterministic_indices(target_table["unit_id"].tolist(), limit, salt + "|T")
    source_sample = source_embeddings[source_index].astype(float)
    target_sample = target_embeddings[target_index].astype(float)
    pooled = np.concatenate([source_sample, target_sample], axis=0)
    domain_labels = np.concatenate([np.zeros(limit, dtype=int), np.ones(limit, dtype=int)])
    components = min(32, pooled.shape[0] - 2, pooled.shape[1])
    projected = PCA(
        n_components=components, svd_solver="randomized", random_state=RANDOM_SEED
    ).fit_transform(pooled)
    classifier = Pipeline([
        ("scale", StandardScaler()),
        ("logistic", LogisticRegression(
            C=1.0, class_weight="balanced", solver="liblinear", random_state=RANDOM_SEED
        )),
    ])
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    probability = cross_val_predict(
        classifier, projected, domain_labels, cv=folds, method="predict_proba"
    )[:, 1]
    domain_auc = float(roc_auc_score(domain_labels, probability))
    squared = cdist(source_sample, source_sample, metric="sqeuclidean")
    positive_distances = squared[np.triu_indices_from(squared, k=1)]
    bandwidth = max(float(np.median(positive_distances[positive_distances > 0])), 1e-8)
    kss = np.exp(-cdist(source_sample, source_sample, metric="sqeuclidean") / bandwidth).mean()
    ktt = np.exp(-cdist(target_sample, target_sample, metric="sqeuclidean") / bandwidth).mean()
    kst = np.exp(-cdist(source_sample, target_sample, metric="sqeuclidean") / bandwidth).mean()
    cosine_cost = cdist(source_sample, target_sample, metric="cosine")
    row_index, column_index = linear_sum_assignment(cosine_cost)
    return {
        "domain_auc": domain_auc,
        "rbf_mmd2": float(kss + ktt - 2 * kst),
        "ot_cosine_cost": float(cosine_cost[row_index, column_index].mean()),
    }


def mixture_components(source_table, target_logits, source_iqr):
    negative = source_table.loc[source_table["label"].eq(0), "logit"].to_numpy(float)
    positive = source_table.loc[source_table["label"].eq(1), "logit"].to_numpy(float)
    target_logits = np.asarray(target_logits, dtype=float)
    values = np.concatenate([negative, positive])
    best = None
    for prevalence in np.linspace(0, 1, 101):
        weights = np.concatenate([
            np.full(len(negative), (1 - prevalence) / len(negative)),
            np.full(len(positive), prevalence / len(positive)),
        ])
        distance = wasserstein_distance(
            target_logits, values,
            u_weights=np.full(len(target_logits), 1 / len(target_logits)),
            v_weights=weights,
        )
        if best is None or distance < best[0]:
            best = (float(distance), float(prevalence))
    return {
        "mixture_wasserstein_residual_normalised": float(best[0] / max(source_iqr, 1e-8)),
        "unlabeled_mixture_prevalence": best[1],
    }


def source_calibration_reference(table):
    probabilities = np.clip(table["probability"].to_numpy(float), 1e-12, 1 - 1e-12)
    labels = table["label"].to_numpy(int)
    predictions = (probabilities >= FIXED_THRESHOLD).astype(int)
    error_rate = 1 - float(np.mean(predictions == labels))
    confidence = np.maximum(probabilities, 1 - probabilities)
    threshold = float(np.quantile(confidence, error_rate))
    return threshold


label_free_rows = []
prediction_rows = []
recoverable_sources = source_recoverability.loc[
    source_recoverability["recoverable"].fillna(False), "source"
].tolist()
for source in recoverable_sources:
    source_asset = domain_assets[source]
    modality = source_asset["modality"]
    source_table_all = source_asset["table"]
    development_mask = source_table_all["partition"].eq("development").to_numpy()
    source_development = source_table_all.loc[development_mask].reset_index(drop=True).copy()
    source_vectors = source_asset["embedding"][development_mask]
    source_probability, source_logits, _ = axis_probability(axes[source], source_vectors)
    source_development["probability"] = source_probability
    source_development["logit"] = source_logits
    source_validation = source_validation_assets[source]
    source_iqr = float(np.subtract(*np.quantile(source_validation["logit"], [0.75, 0.25])))
    atc_threshold = source_calibration_reference(source_validation)

    for target, target_asset in domain_assets.items():
        if target == source or target_asset["modality"] != modality:
            continue
        target_mask = target_asset["table"]["partition"].eq("validation").to_numpy()
        target_table = target_asset["table"].loc[target_mask].reset_index(drop=True).copy()
        target_vectors = target_asset["embedding"][target_mask]
        probabilities, logits, equivalence_error = axis_probability(axes[source], target_vectors)
        support = support_components(source_vectors, target_vectors)
        geometry = geometry_components(
            source_development, source_vectors, target_table, target_vectors, f"{source}|{target}"
        )
        mixture = mixture_components(source_development, logits, source_iqr)
        confidence = np.maximum(probabilities, 1 - probabilities)
        entropy = -(
            probabilities * np.log(np.clip(probabilities, 1e-12, 1)) +
            (1 - probabilities) * np.log(np.clip(1 - probabilities, 1e-12, 1))
        )
        target_iqr = float(np.subtract(*np.quantile(logits, [0.75, 0.25])))
        edge_id = f"{source}__TO__{target}"
        label_free_rows.append({
            "edge_id": edge_id, "modality": modality, "task": target_asset["task"],
            "source": source, "target": target, "target_units": len(target_table),
            "source_validation_auc": float(
                source_recoverability.set_index("source").loc[source, "validation_auc"]
            ),
            "mean_confidence": float(confidence.mean()),
            "mean_entropy_nats": float(entropy.mean()),
            "atc_estimated_accuracy": float(np.mean(confidence >= atc_threshold)),
            "atc_threshold_from_source_validation": atc_threshold,
            "target_to_source_logit_iqr_ratio": float(target_iqr / max(source_iqr, 1e-8)),
            "maximum_axis_equivalence_error": equivalence_error,
            **support, **geometry, **mixture,
        })
        for index, record in target_table.iterrows():
            prediction_rows.append({
                "edge_id": edge_id, "modality": modality, "task": target_asset["task"],
                "source": source, "target": target, "unit_id": record["unit_id"],
                "group_id": record["group_id"], "probability": float(probabilities[index]),
                "logit": float(logits[index]), "images": int(record["images"]),
            })

label_free_columns = [
    "edge_id", "modality", "task", "source", "target", "target_units",
    "source_validation_auc", "mean_confidence", "mean_entropy_nats",
    "atc_estimated_accuracy", "atc_threshold_from_source_validation",
    "target_to_source_logit_iqr_ratio", "maximum_axis_equivalence_error",
    "support_fraction", "fraction_beyond_source_q99", "source_knn_q95",
    "source_knn_q99", "target_mean_knn_distance",
    "target_mean_knn_distance_normalised", "domain_auc", "rbf_mmd2",
    "ot_cosine_cost", "mixture_wasserstein_residual_normalised",
    "unlabeled_mixture_prevalence",
]
prediction_columns = [
    "edge_id", "modality", "task", "source", "target", "unit_id",
    "group_id", "probability", "logit", "images",
]
label_free_edge_table = pd.DataFrame(label_free_rows).reindex(columns=label_free_columns)
frozen_predictions = pd.DataFrame(prediction_rows).reindex(columns=prediction_columns)
LABEL_FREE_EDGE_PATH = FREEZE_ROOT / "Stage8_LabelFree_CrossModality_Edge_Components_v0.1.csv"
PREDICTION_PATH = FREEZE_ROOT / "Stage8_LabelFree_Unit_Predictions_v0.1.csv"
write_csv(LABEL_FREE_EDGE_PATH, label_free_edge_table)
write_csv(PREDICTION_PATH, frozen_predictions)

prediction_freeze_payload = {
    "stage": "Stage8",
    "decision": "CROSS_MODALITY_TARGET_PREDICTIONS_FROZEN_BEFORE_OUTCOME_EVALUATION",
    "protocol_seal_sha256": seal_payload["seal_sha256"],
    "recoverable_sources": recoverable_sources,
    "same_modality_cross_domain_edges": int(len(label_free_edge_table)),
    "label_free_edge_table_sha256": sha256_file(LABEL_FREE_EDGE_PATH),
    "unit_predictions_sha256": sha256_file(PREDICTION_PATH),
    "target_labels_present_in_frozen_prediction_files": False,
    "target_model_refit": False,
    "threshold_tuned": False,
    "final_ddo2_fitted": False,
    "frozen_utc": seal_payload["sealed_utc"],
}
prediction_freeze_payload["freeze_sha256"] = sha256_json(prediction_freeze_payload)
PREDICTION_FREEZE_RECORD_PATH = FREEZE_ROOT / "Stage8_Prediction_Freeze_Complete_v0.1.json"
if PREDICTION_FREEZE_RECORD_PATH.is_file():
    with PREDICTION_FREEZE_RECORD_PATH.open("r", encoding="utf-8") as handle:
        existing = json.load(handle)
    assert existing == prediction_freeze_payload
else:
    atomic_json(PREDICTION_FREEZE_RECORD_PATH, prediction_freeze_payload)

print("Recoverable sources:", recoverable_sources)
print("Frozen same-modality cross-domain edges:", len(label_free_edge_table))
print("Prediction freeze hash:", prediction_freeze_payload["freeze_sha256"])
print("Target labels written into prediction freeze: False")
display(label_free_edge_table)


Recoverable sources: ['ISIC_UDA1', 'ISIC_MSK1', 'HAM10000', 'TBX11K']
Frozen same-modality cross-domain edges: 6
Prediction freeze hash: b24a5ee31b266e401ef60668cfc7c546fdefa9f039ad37dfa2beff3feb9f054f
Target labels written into prediction freeze: False


,edge_id,modality,task,source,target,target_units,source_validation_auc,mean_confidence,mean_entropy_nats,atc_estimated_accuracy,...,fraction_beyond_source_q99,source_knn_q95,source_knn_q99,target_mean_knn_distance,target_mean_knn_distance_normalised,domain_auc,rbf_mmd2,ot_cosine_cost,mixture_wasserstein_residual_normalised,unlabeled_mixture_prevalence
0,ISIC_UDA1__TO__ISIC_MSK1,dermoscopy,melanoma_vs_melanocytic_nevus,ISIC_UDA1,ISIC_MSK1,157,0.720698,0.904606,0.229176,0.738854,...,0.057325,0.314042,0.379770,0.252314,1.276590,0.923157,0.080065,0.338694,0.501161,0.42
1,ISIC_UDA1__TO__HAM10000,dermoscopy,melanoma_vs_melanocytic_nevus,ISIC_UDA1,HAM10000,466,0.720698,0.878239,0.267411,0.660944,...,0.023605,0.314042,0.379770,0.237103,1.199629,0.900940,0.058143,0.271926,0.522188,0.22
2,ISIC_MSK1__TO__ISIC_UDA1,dermoscopy,melanoma_vs_melanocytic_nevus,ISIC_MSK1,ISIC_UDA1,139,0.701161,0.924153,0.185052,0.726619,...,0.021583,0.331331,0.393931,0.241459,1.250820,0.959045,0.098414,0.335842,0.486528,0.08
3,ISIC_MSK1__TO__HAM10000,dermoscopy,melanoma_vs_melanocytic_nevus,ISIC_MSK1,HAM10000,466,0.701161,0.950362,0.126741,0.832618,...,0.019313,0.331331,0.393931,0.241134,1.249139,0.994812,0.078555,0.302069,0.519160,0.05
4,HAM10000__TO__ISIC_UDA1,dermoscopy,melanoma_vs_melanocytic_nevus,HAM10000,ISIC_UDA1,139,0.813744,0.895593,0.223536,0.690647,...,0.043165,0.257699,0.338178,0.201323,1.459999,0.965820,0.082229,0.290112,0.393081,0.32
5,HAM10000__TO__ISIC_MSK1,dermoscopy,melanoma_vs_melanocytic_nevus,HAM10000,ISIC_MSK1,157,0.813744,0.924692,0.171132,0.789809,...,0.089172,0.257699,0.338178,0.229364,1.663356,0.979370,0.090691,0.315606,0.352191,0.42


In [6]:
#@title 08-5. Evaluate frozen predictions, test Stage 7 relation signs, and build the three-modality edge library
from scipy.stats import spearmanr
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix, log_loss,
)


def calibration_error(labels, probabilities, bins=10):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    order = np.argsort(probabilities, kind="mergesort")
    bin_ids = np.empty(len(labels), dtype=int)
    bin_ids[order] = np.minimum(
        np.floor(np.arange(len(labels)) * bins / len(labels)).astype(int), bins - 1
    )
    value = 0.0
    for bin_id in range(bins):
        mask = bin_ids == bin_id
        if np.any(mask):
            value += mask.mean() * abs(probabilities[mask].mean() - labels[mask].mean())
    return float(value)


def outcome_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.clip(np.asarray(probabilities, dtype=float), 1e-12, 1 - 1e-12)
    predictions = (probabilities >= FIXED_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    sensitivity = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    return {
        "auc": float(roc_auc_score(labels, probabilities)),
        "average_precision": float(average_precision_score(labels, probabilities)),
        "brier": float(brier_score_loss(labels, probabilities)),
        "log_loss": float(log_loss(labels, probabilities, labels=[0, 1])),
        "ece10": calibration_error(labels, probabilities),
        "balanced_accuracy": float((sensitivity + specificity) / 2),
        "sensitivity": float(sensitivity), "specificity": float(specificity),
    }


label_lookup = {}
for dataset, asset in domain_assets.items():
    validation = asset["table"][asset["table"]["partition"].eq("validation")]
    label_lookup[dataset] = validation.set_index("unit_id")[["label", "group_id"]]

edge_rows = []
evaluated_predictions = []
source_summary = source_recoverability.set_index("source") if len(source_recoverability) else pd.DataFrame()
for edge in label_free_edge_table.itertuples(index=False):
    target_predictions = frozen_predictions[frozen_predictions["edge_id"].eq(edge.edge_id)].copy()
    labels = label_lookup[edge.target]
    target_predictions = target_predictions.join(labels[["label"]], on="unit_id", how="inner")
    if target_predictions["label"].nunique() != 2:
        continue
    metrics = outcome_metrics(target_predictions["label"], target_predictions["probability"])
    interval_table = target_predictions[["unit_id", "group_id", "label"]].copy()
    interval = patient_bootstrap_auc(
        interval_table, target_predictions["probability"].to_numpy(),
        RANDOM_SEED + len(edge_rows) + 500,
    )
    source_validation = source_validation_assets[edge.source]
    source_metrics = outcome_metrics(source_validation["label"], source_validation["probability"])
    d_pass = bool(metrics["auc"] >= AUC_MINIMUM and interval[0] > AUC_CI_LOWER_STRICT_MINIMUM)
    c_pass = bool(
        metrics["ece10"] <= source_metrics["ece10"] + CALIBRATION_DEGRADATION_TOLERANCE and
        metrics["brier"] <= source_metrics["brier"] + CALIBRATION_DEGRADATION_TOLERANCE
    )
    o_pass = bool(metrics["balanced_accuracy"] >= OPERATING_POINT_BALANCED_ACCURACY_MINIMUM)
    if d_pass and c_pass and o_pass:
        state = "DISCRIMINATION_CALIBRATION_AND_OPERATING_POINT_RETAINED"
    elif d_pass and not c_pass:
        state = "DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED"
    elif d_pass and c_pass and not o_pass:
        state = "DISCRIMINATION_AND_CALIBRATION_RETAINED_BUT_OPERATING_POINT_FAILED"
    else:
        state = "DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIONAL_FAILURES"
    base = label_free_edge_table[label_free_edge_table["edge_id"].eq(edge.edge_id)].iloc[0].to_dict()
    base.update({
        "source_recoverable": True,
        "target_auc": metrics["auc"], "target_auc_ci_lower": interval[0],
        "target_auc_ci_upper": interval[1],
        "source_minus_target_auc": source_metrics["auc"] - metrics["auc"],
        "target_average_precision": metrics["average_precision"],
        "target_brier": metrics["brier"], "target_ece10": metrics["ece10"],
        "calibration_ece_degradation": metrics["ece10"] - source_metrics["ece10"],
        "calibration_brier_degradation": metrics["brier"] - source_metrics["brier"],
        "target_balanced_accuracy_at_0_5": metrics["balanced_accuracy"],
        "target_sensitivity_at_0_5": metrics["sensitivity"],
        "target_specificity_at_0_5": metrics["specificity"],
        "discrimination_pass": d_pass, "calibration_pass": c_pass,
        "operating_point_pass": o_pass,
        "observed_transportability_state": state,
    })
    edge_rows.append(base)
    evaluated_predictions.append(target_predictions)

stage8_edge_columns = list(label_free_edge_table.columns) + [
    "source_recoverable", "target_auc", "target_auc_ci_lower", "target_auc_ci_upper",
    "source_minus_target_auc", "target_average_precision", "target_brier", "target_ece10",
    "calibration_ece_degradation", "calibration_brier_degradation",
    "target_balanced_accuracy_at_0_5", "target_sensitivity_at_0_5",
    "target_specificity_at_0_5", "discrimination_pass", "calibration_pass",
    "operating_point_pass", "observed_transportability_state",
]
stage8_edge_matrix = pd.DataFrame(edge_rows).reindex(columns=list(dict.fromkeys(stage8_edge_columns)))
evaluated_unit_predictions = (
    pd.concat(evaluated_predictions, ignore_index=True) if evaluated_predictions
    else pd.DataFrame()
)
write_csv(DISCOVERY_ROOT / "Stage8_NewModality_Edge_Matrix_v0.1.csv", stage8_edge_matrix)
write_csv(DISCOVERY_ROOT / "Stage8_Evaluated_Unit_Predictions_v0.1.csv", evaluated_unit_predictions)

relations = [
    ("target_auc", "target_mean_knn_distance", -1),
    ("source_minus_target_auc", "atc_estimated_accuracy", 1),
    ("target_ece10", "unlabeled_mixture_prevalence", -1),
    ("calibration_ece_degradation", "atc_estimated_accuracy", 1),
    ("target_balanced_accuracy_at_0_5", "unlabeled_mixture_prevalence", 1),
]
replication_rows = []
for modality in ["dermoscopy", "chest_radiography"]:
    modality_edges = stage8_edge_matrix[stage8_edge_matrix["modality"].eq(modality)]
    for outcome, component, expected_sign in relations:
        subset = modality_edges[[component, outcome]].replace([np.inf, -np.inf], np.nan).dropna()
        rho = (
            float(spearmanr(subset[component], subset[outcome]).statistic)
            if len(subset) >= 3 and subset[component].nunique() > 1 and subset[outcome].nunique() > 1
            else np.nan
        )
        replication_rows.append({
            "modality": modality, "outcome": outcome, "component": component,
            "expected_sign": "positive" if expected_sign > 0 else "negative",
            "n_edges": len(subset), "spearman_rho": rho,
            "sign_replicated": bool(np.isfinite(rho) and np.sign(rho) == expected_sign),
            "status": "DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION",
        })
replication_table = pd.DataFrame(replication_rows)
write_csv(DISCOVERY_ROOT / "Stage8_Stage7_Candidate_Sign_Replication_v0.1.csv", replication_table)

retinal_edges = pd.read_csv(STAGE7_EDGE_PATH)
retinal_edges = retinal_edges[retinal_edges["is_cross_domain"].fillna(False) & retinal_edges["source_recoverable"].fillna(False)].copy()
retinal_edges["modality"] = "retinal_fundus"
retinal_edges["task"] = "referable_diabetic_retinopathy"
retinal_edges["origin_stage"] = "Stage7"
new_edges_for_combination = stage8_edge_matrix.copy()
new_edges_for_combination["origin_stage"] = "Stage8"
common_columns = sorted(set(retinal_edges.columns) | set(new_edges_for_combination.columns))
three_modality_edge_library = pd.concat([
    retinal_edges.reindex(columns=common_columns),
    new_edges_for_combination.reindex(columns=common_columns),
], ignore_index=True)
write_csv(DISCOVERY_ROOT / "Stage8_ThreeModality_Eligible_Edge_Library_v0.1.csv", three_modality_edge_library)

state_summary = (
    stage8_edge_matrix.groupby(["modality", "observed_transportability_state"], as_index=False)
    .agg(edges=("edge_id", "size"), sources=("source", "nunique"), targets=("target", "nunique"))
    if len(stage8_edge_matrix) else pd.DataFrame(columns=["modality", "observed_transportability_state", "edges", "sources", "targets"])
)
write_csv(DISCOVERY_ROOT / "Stage8_NewModality_Transportability_State_Summary_v0.1.csv", state_summary)

print("New-modality evaluated edges:", len(stage8_edge_matrix))
print("Three-modality eligible edge library:", len(three_modality_edge_library))
display(stage8_edge_matrix[[
    "modality", "source", "target", "target_auc", "target_auc_ci_lower", "target_ece10",
    "support_fraction", "domain_auc", "observed_transportability_state",
]] if len(stage8_edge_matrix) else stage8_edge_matrix)
display(replication_table)


New-modality evaluated edges: 6
Three-modality eligible edge library: 15


,modality,source,target,target_auc,target_auc_ci_lower,target_ece10,support_fraction,domain_auc,observed_transportability_state
0,dermoscopy,ISIC_UDA1,ISIC_MSK1,0.596712,0.507769,0.357859,0.815287,0.923157,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
1,dermoscopy,ISIC_UDA1,HAM10000,0.713231,0.646669,0.261493,0.884120,0.900940,DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED
2,dermoscopy,ISIC_MSK1,ISIC_UDA1,0.636364,0.521114,0.190967,0.920863,0.959045,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
3,dermoscopy,ISIC_MSK1,HAM10000,0.763410,0.697269,0.117744,0.929185,0.994812,DISCRIMINATION_AND_CALIBRATION_RETAINED_BUT_OP...
4,dermoscopy,HAM10000,ISIC_UDA1,0.666953,0.548018,0.281214,0.805755,0.965820,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
5,dermoscopy,HAM10000,ISIC_MSK1,0.578143,0.477091,0.371292,0.656051,0.979370,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...


,modality,outcome,component,expected_sign,n_edges,spearman_rho,sign_replicated,status
0,dermoscopy,target_auc,target_mean_knn_distance,negative,6,-0.085714,True,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
1,dermoscopy,source_minus_target_auc,atc_estimated_accuracy,positive,6,-0.028571,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
2,dermoscopy,target_ece10,unlabeled_mixture_prevalence,negative,6,0.985611,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
3,dermoscopy,calibration_ece_degradation,atc_estimated_accuracy,positive,6,-0.028571,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
4,dermoscopy,target_balanced_accuracy_at_0_5,unlabeled_mixture_prevalence,positive,6,0.144943,True,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
5,chest_radiography,target_auc,target_mean_knn_distance,negative,0,NaN,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
6,chest_radiography,source_minus_target_auc,atc_estimated_accuracy,positive,0,NaN,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
7,chest_radiography,target_ece10,unlabeled_mixture_prevalence,negative,0,NaN,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
8,chest_radiography,calibration_ece_degradation,atc_estimated_accuracy,positive,0,NaN,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
9,chest_radiography,target_balanced_accuracy_at_0_5,unlabeled_mixture_prevalence,positive,0,NaN,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION


In [7]:
#@title 08-6. Freeze the Stage 8 decision, integrity manifest, report, figure, and storage audit
import matplotlib.pyplot as plt


ready_by_modality = {}
for modality in ["dermoscopy", "chest_radiography"]:
    registered = DATASET_REGISTRY.loc[DATASET_REGISTRY["modality"].eq(modality), "dataset"]
    ready_by_modality[modality] = sorted(set(registered) & set(domain_assets))
recoverable_by_modality = {}
for modality in ["dermoscopy", "chest_radiography"]:
    recoverable_by_modality[modality] = sorted(
        source_recoverability.loc[
            source_recoverability["modality"].eq(modality) &
            source_recoverability["recoverable"].fillna(False), "source"
        ].tolist()
    )

full_dataset_gate = all(len(ready_by_modality[modality]) >= 3 for modality in ready_by_modality)
source_gate = all(len(recoverable_by_modality[modality]) >= 1 for modality in recoverable_by_modality)
new_edge_gate = len(stage8_edge_matrix) >= 4
three_modality_edge_gate = len(three_modality_edge_library) >= 13
if full_dataset_gate and source_gate and new_edge_gate and three_modality_edge_gate:
    decision = "CROSS_MODAL_EDGE_LIBRARY_ESTABLISHED_FREEZE_DDO2_SPEC_THEN_ACQUIRE_FINAL_HELDOUT_DATASETS"
else:
    decision = "PARTIAL_CROSS_MODAL_EXPANSION_REMEDIATE_ACCESS_OR_SOURCE_RECOVERABILITY_BEFORE_DDO2"

figure_path = RESULT_ROOT / "Stage8_CrossModality_Transportability_Expansion_v0.1.png"
if len(stage8_edge_matrix) and not figure_path.is_file():
    fig, axes_plot = plt.subplots(1, 2, figsize=(11, 4.5))
    colours = {"dermoscopy": "#9c36b5", "chest_radiography": "#1971c2"}
    for modality, subset in stage8_edge_matrix.groupby("modality"):
        axes_plot[0].scatter(
            subset["target_mean_knn_distance_normalised"], subset["target_auc"],
            s=70, alpha=0.85, color=colours.get(modality), label=modality,
        )
        axes_plot[1].scatter(
            subset["unlabeled_mixture_prevalence"], subset["target_ece10"],
            s=70, alpha=0.85, color=colours.get(modality), label=modality,
        )
    axes_plot[0].set(xlabel="Normalised target mean kNN distance", ylabel="Target AUC")
    axes_plot[1].set(xlabel="Unlabelled mixture prevalence", ylabel="Target ECE10")
    for axis_plot in axes_plot:
        axis_plot.grid(alpha=0.2)
        axis_plot.legend(fontsize=8)
    fig.suptitle("Stage 8 cross-modality observability expansion")
    fig.tight_layout()
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    plt.close(fig)

report_lines = [
    "# Stage 8 — Cross-Modality Edge-Library Expansion", "",
    f"- Decision: `{decision}`",
    f"- Ready datasets by modality: `{ready_by_modality}`",
    f"- Recoverable sources by modality: `{recoverable_by_modality}`",
    f"- New same-modality cross-domain edges: `{len(stage8_edge_matrix)}`",
    f"- Eligible retinal + skin + chest edges: `{len(three_modality_edge_library)}`", "",
    "## Interpretation boundary", "",
    "Stage 8 is retrospective discovery expansion. It tests whether the Stage 7 label-free component directions recur in dermoscopy and chest radiography, but it does not fit or validate a final DDO2 predictor.", "",
    "## Data boundary", "",
    "Only official public sources in the frozen registry were attempted. No unavailable dataset was replaced by an unapproved mirror. Original images were streamed or kept in temporary Colab storage and were not copied to Google Drive.", "",
    "## Next gate", "",
    "If both new modalities have at least one recoverable source and the combined edge library is sufficiently populated, freeze a parsimonious hierarchical DDO2 specification before acquiring completely held-out datasets. Otherwise remediate data access or source recoverability without tuning on target outcomes.",
]
report_path = RESULT_ROOT / "Stage8_CrossModality_Expansion_Report_v0.1.md"
report_text = "\n".join(report_lines)
if report_path.is_file():
    assert report_path.read_text(encoding="utf-8") == report_text
else:
    report_path.write_text(report_text, encoding="utf-8")

output_candidates = sorted([
    path for path in STAGE8_ROOT.rglob("*")
    if path.is_file() and path not in {RUNTIME_STATE_PATH, FINAL_RECORD_PATH}
    and "Output_Integrity_Manifest" not in path.name
], key=str)
output_manifest = pd.DataFrame([{
    "relative_path": str(path.relative_to(STAGE8_ROOT)),
    "size_bytes": int(path.stat().st_size), "sha256": sha256_file(path),
} for path in output_candidates])
output_manifest_path = RESULT_ROOT / "Stage8_Output_Integrity_Manifest_v0.1.csv"
write_csv(output_manifest_path, output_manifest)
new_stage8_bytes = int(
    sum(path.stat().st_size for path in output_candidates) + output_manifest_path.stat().st_size
)
assert new_stage8_bytes <= MAXIMUM_NEW_STAGE8_BYTES, (
    f"Stage 8 exceeded its 1 GiB Drive cap: {new_stage8_bytes} bytes"
)

final_payload = {
    "stage": "Stage8", "decision": decision,
    "scope": "RETROSPECTIVE_CROSS_MODALITY_DISCOVERY_EXPANSION_NOT_FINAL_DDO2_VALIDATION",
    "stage7_final_record_sha256": EXPECTED_STAGE7_FINAL_HASH,
    "stage8_protocol_seal_sha256": seal_payload["seal_sha256"],
    "prediction_freeze_sha256": prediction_freeze_payload["freeze_sha256"],
    "ready_datasets_by_modality": ready_by_modality,
    "recoverable_sources_by_modality": recoverable_by_modality,
    "new_modality_edges": int(len(stage8_edge_matrix)),
    "three_modality_eligible_edges": int(len(three_modality_edge_library)),
    "stage7_candidate_sign_replication": json.loads(replication_table.to_json(orient="records")),
    "new_modality_state_counts": (
        stage8_edge_matrix["observed_transportability_state"].value_counts().astype(int).to_dict()
        if len(stage8_edge_matrix) else {}
    ),
    "external_data_downloaded": True,
    "image_files_read": bool(len(embedding_audit)),
    "image_files_copied_to_drive": False,
    "target_model_refit": False, "threshold_tuned": False, "final_ddo2_fitted": False,
    "new_stage8_bytes": new_stage8_bytes,
    "maximum_new_stage8_bytes": MAXIMUM_NEW_STAGE8_BYTES,
    "output_integrity_manifest_sha256": sha256_file(output_manifest_path),
    "next_step": (
        "FREEZE_HIERARCHICAL_DDO2_SPECIFICATION_AND_ACQUIRE_FINAL_HELDOUT_DATASETS"
        if decision.startswith("CROSS_MODAL_EDGE_LIBRARY_ESTABLISHED")
        else "REMEDIATE_DATA_ACCESS_OR_SOURCE_RECOVERABILITY_WITHOUT_TARGET_TUNING"
    ),
    "completed_utc": seal_payload["sealed_utc"],
}
final_payload["final_record_sha256"] = sha256_json(final_payload)
if FINAL_RECORD_PATH.is_file():
    with FINAL_RECORD_PATH.open("r", encoding="utf-8") as handle:
        existing = json.load(handle)
    assert existing == final_payload
else:
    atomic_json(FINAL_RECORD_PATH, final_payload)

runtime_state.update({
    "stage8_complete": True, "decision": decision,
    "external_data_downloaded": True, "image_files_read": bool(len(embedding_audit)),
    "image_files_copied_to_drive": False, "target_model_refit": False,
    "threshold_tuned": False, "final_ddo2_fitted": False,
    "new_stage8_bytes": new_stage8_bytes,
    "final_record_sha256": final_payload["final_record_sha256"],
    "last_updated_utc": utc_now(),
})
atomic_json(RUNTIME_STATE_PATH, runtime_state)

# Delete only the explicit temporary Stage 8 working directory; no Drive images exist.
if TEMP_ROOT.is_dir():
    shutil.rmtree(TEMP_ROOT)

print("\n================ STAGE 8 CROSS-MODAL EXPANSION COMPLETE ================")
display(source_recoverability)
display(stage8_edge_matrix[[
    "modality", "source", "target", "target_auc", "target_auc_ci_lower",
    "target_ece10", "target_balanced_accuracy_at_0_5", "support_fraction",
    "domain_auc", "observed_transportability_state",
]] if len(stage8_edge_matrix) else stage8_edge_matrix)
display(replication_table)
print("\nDecision:", decision)
print("Final record:", FINAL_RECORD_PATH)
print("Final record hash:", final_payload["final_record_sha256"])
print("New Stage 8 Drive storage (MiB):", new_stage8_bytes / 1024**2)
print("Storage cap (MiB):", MAXIMUM_NEW_STAGE8_BYTES / 1024**2)
print("Original images copied to Drive: False")
print("Target refit / threshold tuning / final DDO2 fit: False / False / False")
print("\nSTOP. Interpret Stage 8 before freezing a final DDO2 specification or acquiring held-out validation datasets.")



================ STAGE 8 CROSS-MODAL EXPANSION COMPLETE ================


,source,modality,development_units,validation_units,development_oof_auc,development_oof_auc_ci_lower,development_oof_auc_ci_upper,validation_auc,validation_auc_ci_lower,validation_auc_ci_upper,recoverable,failure_reason,axis_path,axis_sha256,maximum_axis_equivalence_error
0,ISIC_UDA1,dermoscopy,415,139,0.747240,0.693216,0.794872,0.720698,0.600825,0.831265,True,,03_Frozen_Source_Axes/ISIC_UDA1_Frozen_Source_...,316857360dd3a4f3c9f19f811cc58a4c17c1da1372319c...,1.598721e-14
1,ISIC_MSK1,dermoscopy,489,157,0.773746,0.725815,0.820006,0.701161,0.603893,0.787039,True,,03_Frozen_Source_Axes/ISIC_MSK1_Frozen_Source_...,5011be88de7e877d836f0518540c53dd3f6a34c57225b7...,1.598721e-14
2,HAM10000,dermoscopy,1398,466,0.850231,0.824684,0.877929,0.813744,0.749970,0.875653,True,,03_Frozen_Source_Axes/HAM10000_Frozen_Source_A...,0988518fcfbcb3436f43fbdff37b61395b8ba8b3715fcc...,2.842171e-14
3,TBX11K,chest_radiography,1500,500,0.958827,0.940773,0.975102,0.963788,0.939730,0.983316,True,,03_Frozen_Source_Axes/TBX11K_Frozen_Source_Axi...,6fc63f746731f607173d7aff4ffdfb2e580be2a4abe5b6...,1.643130e-14


,modality,source,target,target_auc,target_auc_ci_lower,target_ece10,target_balanced_accuracy_at_0_5,support_fraction,domain_auc,observed_transportability_state
0,dermoscopy,ISIC_UDA1,ISIC_MSK1,0.596712,0.507769,0.357859,0.608607,0.815287,0.923157,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
1,dermoscopy,ISIC_UDA1,HAM10000,0.713231,0.646669,0.261493,0.659515,0.884120,0.900940,DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED
2,dermoscopy,ISIC_MSK1,ISIC_UDA1,0.636364,0.521114,0.190967,0.535306,0.920863,0.959045,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
3,dermoscopy,ISIC_MSK1,HAM10000,0.763410,0.697269,0.117744,0.587687,0.929185,0.994812,DISCRIMINATION_AND_CALIBRATION_RETAINED_BUT_OP...
4,dermoscopy,HAM10000,ISIC_UDA1,0.666953,0.548018,0.281214,0.637936,0.805755,0.965820,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
5,dermoscopy,HAM10000,ISIC_MSK1,0.578143,0.477091,0.371292,0.547776,0.656051,0.979370,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...


,modality,outcome,component,expected_sign,n_edges,spearman_rho,sign_replicated,status
0,dermoscopy,target_auc,target_mean_knn_distance,negative,6,-0.085714,True,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
1,dermoscopy,source_minus_target_auc,atc_estimated_accuracy,positive,6,-0.028571,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
2,dermoscopy,target_ece10,unlabeled_mixture_prevalence,negative,6,0.985611,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
3,dermoscopy,calibration_ece_degradation,atc_estimated_accuracy,positive,6,-0.028571,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
4,dermoscopy,target_balanced_accuracy_at_0_5,unlabeled_mixture_prevalence,positive,6,0.144943,True,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
5,chest_radiography,target_auc,target_mean_knn_distance,negative,0,NaN,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
6,chest_radiography,source_minus_target_auc,atc_estimated_accuracy,positive,0,NaN,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
7,chest_radiography,target_ece10,unlabeled_mixture_prevalence,negative,0,NaN,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
8,chest_radiography,calibration_ece_degradation,atc_estimated_accuracy,positive,0,NaN,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
9,chest_radiography,target_balanced_accuracy_at_0_5,unlabeled_mixture_prevalence,positive,0,NaN,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION



Decision: PARTIAL_CROSS_MODAL_EXPANSION_REMEDIATE_ACCESS_OR_SOURCE_RECOVERABILITY_BEFORE_DDO2
Final record: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Cross_Modal/Stage8_CrossModality_EdgeLibrary_Expansion_v0.1/06_Results/Stage8_CrossModality_Expansion_Complete_v0.1.json
Final record hash: 698c019b8516a84522831216b6cb0c85e047e6d0b401071db99e1d74ae6b2397
New Stage 8 Drive storage (MiB): 30.355647087097168
Storage cap (MiB): 1024.0
Original images copied to Drive: False
Target refit / threshold tuning / final DDO2 fit: False / False / False

STOP. Interpret Stage 8 before freezing a final DDO2 specification or acquiring held-out validation datasets.
